# 📊 Notebook 05: Exploratory Data Analysis & Visualization

**Objective:** Deep exploration of the consolidated drought & water stress dataset.

**Contents:**
1. Dataset overview & statistics
2. Global water stress heatmaps
3. Time series of TWS anomalies
4. Correlation analysis
5. Distribution analysis
6. Seasonal decomposition

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import config

plt.style.use('dark_background')
sns.set_theme(style='darkgrid', palette='viridis')
print('✅ Setup complete')

In [ ]:
# Load consolidated data
df = pd.read_csv(config.CONSOLIDATED_CSV)
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nDate range: {df["date"].min()} to {df["date"].max()}' if 'date' in df.columns else '')
print(f'Countries: {df["country_code"].nunique()}')
df.head()

## 1. Dataset Overview

In [ ]:
# Descriptive statistics
df.describe().round(3)

In [ ]:
# Missing value analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Percent', ascending=False)
if len(missing_df) > 0:
    print('Missing values:')
    print(missing_df)
else:
    print('✅ No missing values')

## 2. Global Water Stress Heatmaps

In [ ]:
# Choropleth: Drought Composite Score
score_col = 'drought_composite_score' if 'drought_composite_score' in df.columns else 'water_stress_score'
country_avg = df.groupby(['country_code', 'country']).agg({score_col: 'mean'}).reset_index()

fig = px.choropleth(
    country_avg, locations='country_code', locationmode='ISO-3',
    color=score_col, hover_name='country',
    color_continuous_scale='RdYlBu_r',
    title=f'Global {score_col.replace("_", " ").title()}',
)
fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
    template='plotly_dark', height=550,
)
fig.show()

In [ ]:
# Drought Risk Class Distribution
if 'drought_risk_class' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Pie chart
    colors = {'Low': '#2ecc71', 'Moderate': '#f39c12', 'High': '#e74c3c', 'Extreme': '#8e44ad'}
    dist = df['drought_risk_class'].value_counts()
    axes[0].pie(dist.values, labels=dist.index, autopct='%1.1f%%',
                colors=[colors.get(l, '#999') for l in dist.index],
                textprops={'fontsize': 11})
    axes[0].set_title('Drought Risk Distribution', fontsize=13, fontweight='bold')
    
    # By country
    country_risk = df.groupby(['country', 'drought_risk_class']).size().unstack(fill_value=0)
    top_countries = df.groupby('country')[score_col].mean().nlargest(15).index
    country_risk_top = country_risk.loc[country_risk.index.isin(top_countries)]
    
    if not country_risk_top.empty:
        ordered = [c for c in ['Low', 'Moderate', 'High', 'Extreme'] if c in country_risk_top.columns]
        bar_colors = [colors.get(c, '#999') for c in ordered]
        country_risk_top[ordered].plot(kind='barh', stacked=True, ax=axes[1], color=bar_colors)
        axes[1].set_title('Risk Distribution — Top 15 Countries', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Count')
    
    plt.tight_layout()
    plt.show()

## 3. Time Series Analysis

In [ ]:
# Global average TWS anomaly over time
if 'tws_anomaly_cm' in df.columns and 'date' in df.columns:
    global_avg = df.groupby('date').agg({
        'tws_anomaly_cm': 'mean',
        'drought_composite_score': 'mean'
    }).reset_index()
    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['Global Average TWS Anomaly', 'Global Drought Composite Score'])
    
    fig.add_trace(
        go.Scatter(x=global_avg['date'], y=global_avg['tws_anomaly_cm'],
                   fill='tozeroy', name='TWS Anomaly',
                   line=dict(color='#3498db', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=global_avg['date'], y=global_avg['drought_composite_score'],
                   name='Drought Score', line=dict(color='#e74c3c', width=2)),
        row=2, col=1
    )
    
    fig.update_layout(template='plotly_dark', height=500, showlegend=True)
    fig.show()

## 4. Correlation Analysis

In [ ]:
# Correlation matrix
numeric_cols = df.select_dtypes(include=[np.number]).columns
key_cols = [c for c in numeric_cols if any(
    kw in c.lower() for kw in ['stress', 'drought', 'tws', 'water', 'ground', 'precip', 'composite']
)]

if len(key_cols) >= 3:
    corr = df[key_cols].corr()
    
    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
                center=0, square=True, linewidths=0.5, ax=ax)
    ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Distribution Analysis

In [ ]:
# KDE plots for key indicators
plot_cols = [c for c in ['drought_composite_score', 'tws_anomaly_cm', 
                          'water_stress_score', 'groundwater_anomaly_cm'] if c in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#667eea', '#e74c3c', '#f39c12', '#2ecc71']

for i, (col, color) in enumerate(zip(plot_cols, colors)):
    ax = axes[i // 2, i % 2]
    data = df[col].dropna()
    ax.hist(data, bins=50, color=color, alpha=0.6, edgecolor='white', linewidth=0.3, density=True)
    data.plot.kde(ax=ax, color='white', linewidth=2)
    ax.set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.axvline(data.mean(), color='yellow', linestyle='--', label=f'μ={data.mean():.2f}')
    ax.axvline(data.median(), color='cyan', linestyle=':', label=f'Median={data.median():.2f}')
    ax.legend(fontsize=9)

plt.suptitle('Distribution of Key Indicators', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Seasonal Patterns

In [ ]:
# Seasonal box plots
if 'season' in df.columns and 'drought_composite_score' in df.columns:
    fig = px.box(
        df, x='season', y='drought_composite_score', color='season',
        category_orders={'season': ['DJF', 'MAM', 'JJA', 'SON']},
        color_discrete_map={'DJF': '#3498db', 'MAM': '#2ecc71', 'JJA': '#e74c3c', 'SON': '#f39c12'},
        title='Drought Composite Score by Season',
    )
    fig.update_layout(template='plotly_dark', height=450, showlegend=False)
    fig.show()

# Monthly patterns
if 'date' in df.columns and 'tws_anomaly_cm' in df.columns:
    df['month'] = df['date'].dt.month
    monthly = df.groupby('month')['tws_anomaly_cm'].agg(['mean', 'std']).reset_index()
    
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=monthly['month'], y=monthly['mean'],
        error_y=dict(type='data', array=monthly['std'], visible=True),
        marker_color=['#3498db' if v >= 0 else '#e74c3c' for v in monthly['mean']],
        name='Mean TWS Anomaly'
    ))
    fig.update_layout(
        title='Monthly TWS Anomaly Pattern (Global Average)',
        xaxis_title='Month', yaxis_title='TWS Anomaly (cm)',
        template='plotly_dark', height=400,
    )
    fig.show()

In [ ]:
print('\n✅ EDA Complete')
print(f'Dataset: {config.CONSOLIDATED_CSV}')
print(f'Records: {len(df):,}')
print(f'Features: {len(df.columns)}')